# Advanced RAG with Local Qwen3 4B + DeepEval

**Environment:** Python 3.12  
**LangChain:** 1.3.x  
**Evaluation:** DeepEval  
**Generator / evaluator:** local `Qwen3 4B` through Ollama

### What this notebook does

This notebook builds an Advanced RAG pipeline:

**PDF → Markdown → clean text → real headings → structure-aware semantic chunks → ChromaDB → dense retrieval + BM25 → hybrid retrieval → cross-encoder reranking → Qwen3 4B generation → DeepEval evaluation**

### Local LLM setup

Qwen3 4B is served locally by Ollama. The same local model is used for:
1. RAG answer generation
2. DeepEval evaluation

No OpenAI API is required by the notebook.

> Before running the notebook, install Qwen3 4B with:
> `ollama pull qwen3:4b`

DeepEval supports Ollama models directly, so the evaluation path stays local as well.


## 1. Configuration

Keep paths and retrieval parameters in one place so the rest of the notebook is easy to follow.

In [29]:
from pathlib import Path
import json
import re
import numpy as np

RAW_PDF = Path("data/raw/MachineLearningTomMitchell.pdf")
MD_PATH = Path("data/processed/ml_book.md")
CLEAN_PATH = Path("data/processed/clean_ml_book.md")
CHUNKS_PATH = Path("data/processed/chunks.jsonl")
CHROMA_PATH = "data/chroma"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Ollama model used for both generation and DeepEval judging.
LOCAL_MODEL = "qwen3:4b-instruct"
OLLAMA_BASE_URL = "http://localhost:11434"

TOP_K_HYBRID = 8
TOP_K_FINAL = 4

MD_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("LLM:", LOCAL_MODEL)


Configuration loaded.
LLM: qwen3:4b-instruct


## 2. Check the environment

This cell only checks the Python environment and installed package versions.

The notebook does not configure or call OpenAI.


In [2]:
import sys
import langchain
import deepeval

print("Python:", sys.version)
print("LangChain:", langchain.__version__)
print("DeepEval:", deepeval.__version__)

assert sys.version_info[:2] == (3, 12), "This notebook was prepared for Python 3.12."

print("Environment check completed.")


Python: 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]
LangChain: 1.3.15
DeepEval: 4.1.8
Environment check completed.


## 3. PDF → Markdown

The extraction is cached. If the Markdown file already exists, the expensive PDF extraction is skipped.

In [3]:
if MD_PATH.exists():
    print(f"Already extracted: {MD_PATH}")
else:
    import pymupdf
    import pymupdf4llm
    from tqdm import tqdm

    doc = pymupdf.open(RAW_PDF)
    print(f"Total pages: {len(doc)}")

    all_pages = []
    for page_num in tqdm(range(len(doc)), desc="Extracting PDF", unit="page"):
        page_data = pymupdf4llm.to_markdown(
            doc,
            pages=[page_num],
            page_chunks=True
        )
        all_pages.extend(page_data)

    doc.close()

    with open(MD_PATH, "w", encoding="utf-8") as f:
        for page_number, page in enumerate(all_pages, start=1):
            f.write(f"\n\n<!-- PAGE {page_number} -->\n\n")
            f.write(page["text"])

    print(f"Saved to: {MD_PATH}")

Already extracted: data\processed\ml_book.md


In [4]:
text = MD_PATH.read_text(encoding="utf-8")

pages = re.findall(r"<!-- PAGE (\d+) -->", text)

print(f"Characters: {len(text):,}")
print(f"Words: {len(text.split()):,}")
print(f"Page markers: {len(pages)}")

Characters: 1,134,785
Words: 174,545
Page markers: 421


## 4. Clean the extracted Markdown

We keep the page markers because page numbers are useful metadata later when showing the source of an answer.

In [5]:
if CLEAN_PATH.exists():
    print(f"Already cleaned: {CLEAN_PATH}")
else:
    text = MD_PATH.read_text(encoding="utf-8")

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\n*\s*(<!-- PAGE \d+ -->)\s*\n*", r"\n\n\1\n\n", text)
    text = re.sub(r" +([,.!?;:])", r"\1", text)

    CLEAN_PATH.write_text(text.strip(), encoding="utf-8")
    print(f"Saved to: {CLEAN_PATH}")

text = CLEAN_PATH.read_text(encoding="utf-8")
print(f"Cleaned characters: {len(text):,}")

Already cleaned: data\processed\clean_ml_book.md
Cleaned characters: 1,131,404


## 5. Keep only real section headings

`pymupdf4llm` can turn bold text, captions and other emphasized text into Markdown headings.

We therefore keep:
- numbered section headings such as `3.7.1 ...`
- genuine standalone ALL-CAPS headings

Figure and table captions are explicitly rejected.

In [6]:
HEADING_RE = re.compile(r"(?m)^(#{1,6})\s+(.+?)\s*$")
PAGE_RE = re.compile(r"<!-- PAGE (\d+) -->")

def strip_md_emphasis(s: str) -> str:
    return re.sub(r"[*_]+", "", s).strip()

def is_real_heading(raw_title: str) -> bool:
    title = strip_md_emphasis(raw_title)

    if re.match(r"^(FIGURE|TABLE)\b", title, re.IGNORECASE):
        return False

    if re.match(r"^\d+(\.\d+){0,4}\s+\S", title):
        return True

    letters_only = re.sub(r"[^A-Za-z]", "", title)
    if len(letters_only) >= 4 and letters_only.isupper():
        return True

    return False

structure = []
current_page = None

for line in text.splitlines():
    page_match = PAGE_RE.match(line.strip())
    if page_match:
        current_page = int(page_match.group(1))
        continue

    heading_match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line.strip())
    if heading_match:
        level = len(heading_match.group(1))
        raw_title = heading_match.group(2)

        if is_real_heading(raw_title):
            structure.append({
                "page": current_page,
                "level": level,
                "title": strip_md_emphasis(raw_title),
            })

print(f"Real headings kept: {len(structure)}")

for item in structure[:20]:
    print(f"p{item['page']:>3} | {'  ' * (item['level'] - 1)}{item['title']}")

Real headings kept: 298
p  3 | PREFACE
p  4 | ACKNOWLEDGMENTS
p 14 | 1.1 WELL-POSED LEARNING PROBLEMS
p 17 | 1.2 DESIGNING A LEARNING SYSTEM
p 17 | 1.2.1 Choosing the Training Experience
p 19 | 1.2.2 Choosing the Target Function
p 20 | 1.23 Choosing a Representation for the Target Function
p 21 | 1.2.4 Choosing a Function Approximation Algorithm
p 22 | 1.2.4.1 ESTIMATING TRAINING VALUES
p 22 | 1.2.4.2 ADJUSTING THE WEIGHTS
p 23 | 1.2.5 The Final Design
p 26 | 1.3 PERSPECTIVES AND ISSUES IN MACHINE LEARNING
p 27 | 1.3.1 Issues in Machine Learning
p 28 | 1.4 HOW TO READ THIS BOOK
p 29 | 1.5 SUMMARY AND FURTHER READING
p 30 | EXERCISES
p 31 | REFERENCES
p 32 | 2.1 INTRODUCTION
p 33 | 2.2 A CONCEPT LEARNING TASK
p 34 | 2.2.1 Notation


## 6. Split the book into structure-aware sections

Each section keeps its heading path and starting page. This metadata will later be attached to every retrieved document.

In [7]:
def find_heading_positions(text, structure):
    positions = []
    search_from = 0

    for item in structure:
        pattern = re.compile(
            r"(?m)^#{1,6}\s+\*{0,3}_{0,3}" +
            re.escape(item["title"][:40])
        )
        match = pattern.search(text, search_from)

        if match is None:
            match = pattern.search(text)

        positions.append(match.start() if match else search_from)

        if match:
            search_from = match.end()

    return positions

def heading_path(idx):
    level = structure[idx]["level"]
    path = [structure[idx]["title"]]

    for j in range(idx - 1, -1, -1):
        if structure[j]["level"] < level:
            path.insert(0, structure[j]["title"])
            level = structure[j]["level"]

        if level <= 1:
            break

    return " > ".join(path)

positions = find_heading_positions(text, structure)

sections = []

for i, item in enumerate(structure):
    start = positions[i]
    end = positions[i + 1] if i + 1 < len(positions) else len(text)

    body = text[start:end]
    body = re.sub(r"(?m)^#{1,6}\s+.+?$", "", body, count=1).strip()
    body = PAGE_RE.sub("", body).strip()

    if body:
        sections.append({
            "title": item["title"],
            "heading_path": heading_path(i),
            "page": item["page"],
            "text": body,
        })

print(f"Sections with body text: {len(sections)}")

Sections with body text: 280


## 7. Semantic chunking

We use LangChain's `SemanticChunker` inside each real section.

A fallback recursive splitter is used before semantic chunking for very large sections. This prevents one enormous section from becoming an unnecessarily expensive embedding operation.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

import torch

embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", embedding_device)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": embedding_device},
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

fallback_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

MIN_CHUNK_CHARS = 200
MAX_SEMANTIC_INPUT = 20000

W0816 22:13:56.576000 22876 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\Swapn\AppData\Local\Temp\ipykernel_22876\2100202115.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Embedding device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
merged_sections = []
carry = ""

for sec in sections:
    combined = (carry + "\n\n" + sec["text"]).strip() if carry else sec["text"]

    if len(combined) < MIN_CHUNK_CHARS:
        carry = combined
        continue

    merged_sections.append({**sec, "text": combined})
    carry = ""

if carry:
    if merged_sections:
        merged_sections[-1]["text"] += "\n\n" + carry
    else:
        merged_sections.append({**sections[-1], "text": carry})

print(
    f"Sections before merge: {len(sections)} -> "
    f"after merge: {len(merged_sections)}"
)

Sections before merge: 280 -> after merge: 280


In [10]:
def chunk_section_text(section_text):
    pieces = (
        fallback_splitter.split_text(section_text)
        if len(section_text) > MAX_SEMANTIC_INPUT
        else [section_text]
    )

    chunks_out = []

    for piece in pieces:
        chunks_out.extend(semantic_splitter.split_text(piece))

    return chunks_out

chunks = []

for sec in merged_sections:
    for piece in chunk_section_text(sec["text"]):
        piece = piece.strip()

        if not piece:
            continue

        chunks.append({
            "chunk_id": len(chunks),
            "heading_path": sec["heading_path"],
            "page": sec["page"],
            "text": piece,
            "n_chars": len(piece),
        })

print(f"Total chunks: {len(chunks):,}")

lengths = [c["n_chars"] for c in chunks]
print(f"Min: {min(lengths)}")
print(f"Max: {max(lengths)}")
print(f"Average: {sum(lengths) / len(lengths):.0f}")

Total chunks: 1,234
Min: 1
Max: 6809
Average: 897


## 8. Save the chunks

Saving the chunks lets us rebuild the vector database without repeating PDF extraction and semantic chunking.

In [11]:
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Saved {len(chunks):,} chunks to {CHUNKS_PATH}")

Saved 1,234 chunks to data\processed\chunks.jsonl


## 9. Create LangChain Documents

ChromaDB stores the chunk text while the metadata keeps the section and page information.

In [12]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=chunk["text"],
        metadata={
            "chunk_id": chunk["chunk_id"],
            "heading_path": chunk["heading_path"],
            "page": chunk["page"],
        },
    )
    for chunk in chunks
]

print(f"Documents ready: {len(documents):,}")

Documents ready: 1,234


## 10. Build the ChromaDB vector store

Dense retrieval finds chunks that are semantically similar to the query.

In [13]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ml_book",
    persist_directory=CHROMA_PATH,
)

print("ChromaDB created successfully.")

ChromaDB created successfully.


## 11. Dense retriever

In [14]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_HYBRID},
)

query = "What is machine learning?"
dense_docs = retriever.invoke(query)

for i, doc in enumerate(dense_docs, 1):
    print(f"\n--- Dense result {i} ---")
    print(doc.page_content[:350])
    print(doc.metadata)


--- Dense result 1 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where 
{'heading_path': '1.5 SUMMARY AND FURTHER READING', 'chunk_id': 48, 'page': 29}

--- Dense result 2 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where 
{'page': 29, 'chunk_id': 48, 'heading_path': '1.5 SUMMARY AND FURTHER READING'}

--- Dense result 3 ---
The field of machine learning is concerned with the question of ho

## 12. BM25 lexical retrieval

Dense retrieval is good for meaning, while BM25 is good at exact terms.

We combine both later with Reciprocal Rank Fusion (RRF).

In [15]:
from rank_bm25 import BM25Okapi

tokenized_docs = [
    doc.page_content.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_docs)

print("BM25 index created.")

BM25 index created.


## 13. Hybrid search with Reciprocal Rank Fusion

A document receives a score from both dense retrieval and BM25. RRF combines their rankings without requiring the two raw scores to be comparable.

In [16]:
def hybrid_search(query, k=TOP_K_HYBRID, fetch_k=20):
    dense_docs = retriever.invoke(query)[:fetch_k]

    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_indices = bm25_scores.argsort()[-fetch_k:][::-1]

    rrf_scores = {}

    for rank, doc in enumerate(dense_docs):
        doc_id = doc.metadata["chunk_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (60 + rank + 1)

    for rank, idx in enumerate(bm25_indices):
        doc_id = documents[idx].metadata["chunk_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (60 + rank + 1)

    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:k]

    doc_lookup = {
        doc.metadata["chunk_id"]: doc
        for doc in documents
    }

    return [doc_lookup[doc_id] for doc_id in ranked_ids]

In [17]:
results = hybrid_search("What is machine learning?", k=5)

for i, doc in enumerate(results, 1):
    print(f"\n--- Hybrid result {i} ---")
    print(doc.page_content[:350])


--- Hybrid result 1 ---
The field of machine learning is concerned with the question of how to construct computer programs that automatically improve with experience. In recent years many successful machine learning applications have been developed, ranging from data-mining programs that learn to detect fraudulent credit card transactions, to information-filtering systems

--- Hybrid result 2 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where 

--- Hybrid result 3 ---
This book contains an introduction to the primary algorithms and approaches to machine learning, theoretical results on the feasibility of various learning tasks and the capabilities of specific algorithms, and examples of 

## 14. Cross-encoder reranking

Hybrid search gives us a candidate set. The cross-encoder reads the query and each candidate together and produces a more direct relevance score.

We retrieve more candidates first, then keep the best few.

In [18]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANKER_MODEL)

def rerank(query, docs, top_k=TOP_K_FINAL):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for _, doc in ranked[:top_k]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
query = "What is machine learning?"

hybrid_docs = hybrid_search(query, k=TOP_K_HYBRID)
final_docs = rerank(query, hybrid_docs, top_k=TOP_K_FINAL)

for i, doc in enumerate(final_docs, 1):
    print(f"\n--- Final context {i} ---")
    print(doc.page_content[:400])
    print(doc.metadata)


--- Final context 1 ---
The field of machine learning is concerned with the question of how to construct computer programs that automatically improve with experience. In recent years many successful machine learning applications have been developed, ranging from data-mining programs that learn to detect fraudulent credit card transactions, to information-filtering systems that learn users' reading preferences, to autonom
{'chunk_id': 0, 'heading_path': 'PREFACE', 'page': 3}

--- Final context 2 ---
Machine learning addresses the question of how to build computer programs that improve their performance at some task through experience. Major points of this chapter include: 

 - Machine learning algorithms have proven to be of great practical value in a variety of application domains. They are especially useful in (a) data mining problems where large databases may contain valuable implicit regu
{'chunk_id': 48, 'heading_path': '1.5 SUMMARY AND FURTHER READING', 'page': 29}

--- Final con

## 15. Build the RAG prompt

The generator receives only the reranked context.

The source tag is included so Qwen3 can cite the section/page when it answers.


In [34]:
def build_context(docs):
    parts = []

    for doc in docs:
        meta = doc.metadata
        source = (
            f"[{meta.get('heading_path', 'Unknown section')}, "
            f"p.{meta.get('page', '?')}]"
        )
        parts.append(f"{source}\n{doc.page_content}")

    return "\n\n".join(parts)


def build_prompt(context, question):
    return f"""<|system|>
You are a helpful question-answering assistant for machine learning.

Answer the question using ONLY the supplied context.

Give a clear and sufficiently detailed answer. Use multiple sentences
when the context provides useful supporting information.

Do not add information that is not supported by the context.

Cite the relevant source tag at the end of the answer when appropriate.

If the answer is not present in the context, say:
"I don't know based on the provided context."

<|user|>
Context:
{context}

Question:
{question}

<|assistant|>
"""


## 16. Connect Qwen3 4B through Ollama

Ollama keeps the generator local. The model is loaded and managed by the Ollama service rather than by `transformers`.

This is simpler than the previous TinyLlama setup and avoids loading a second copy of the model inside the notebook.


In [35]:
from langchain_ollama import ChatOllama

local_llm = ChatOllama(
    model=LOCAL_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
    num_predict=512,
    reasoning=False
)

print(f"Connected to Ollama model: {LOCAL_MODEL}")
print(f"Ollama URL: {OLLAMA_BASE_URL}")


Connected to Ollama model: qwen3:4b-instruct
Ollama URL: http://localhost:11434


In [36]:
test_response = local_llm.invoke(
    "Explain machine learning in one sentence."
)

print("CONTENT:")
print(test_response.content)

print("\nADDITIONAL KWARGS:")
print(test_response.additional_kwargs)

CONTENT:
Machine learning is a field of artificial intelligence that enables computers to learn from data and improve their performance on tasks without being explicitly programmed.

ADDITIONAL KWARGS:
{}


## 17. Complete RAG function

The complete flow is now:

**query → hybrid retrieval → reranking → context → Qwen3 4B**


In [37]:
def extract_response_text(response):
    content = response.content

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        texts = []

        for block in content:
            if isinstance(block, dict) and "text" in block:
                texts.append(block["text"])

        return "".join(texts).strip()

    return str(content).strip()


def rag(query):
    hybrid_docs = hybrid_search(
        query,
        k=TOP_K_HYBRID,
        fetch_k=20,
    )

    final_docs = rerank(
        query,
        hybrid_docs,
        top_k=TOP_K_FINAL,
    )

    context = build_context(final_docs)
    prompt_text = build_prompt(context, query)

    response = local_llm.invoke(prompt_text)
    answer = extract_response_text(response)

    return answer, final_docs


In [38]:
question = "What is machine learning?"

answer, sources = rag(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

print("\nSOURCES:")
for i, doc in enumerate(sources, 1):
    print(f"{i}. {doc.metadata}")


QUESTION:
What is machine learning?

ANSWER:
Machine learning is the field concerned with constructing computer programs that automatically improve their performance at a specific task through experience. It involves developing algorithms that can learn general target functions from specific training examples and adapt over time based on data. These programs are particularly valuable in data-mining applications where implicit patterns in large databases can be discovered, in domains where human knowledge is insufficient to design effective algorithms, and in situations where systems must dynamically adapt to changing conditions. The field draws on concepts from various disciplines such as statistics, artificial intelligence, information theory, cognitive science, and philosophy, and it seeks to understand both the algorithms and the theoretical foundations that govern learning processes. [PREFACE, p.3]

SOURCES:
1. {'chunk_id': 0, 'heading_path': 'PREFACE', 'page': 3}
2. {'chunk_id': 4

## 18. Prepare a small evaluation set

DeepEval uses an `LLMTestCase` containing the question, generated answer, retrieved context and, for reference-based retrieval metrics, an expected answer.

The expected answers below are short reference answers for this learning project. They are **evaluation references**, not the same thing as the evaluator model.


In [39]:
eval_set = [
    {
        "question": "What is machine learning?",
        "reference": (
            "Machine learning is the study of computer programs that "
            "improve their performance on a task through experience."
        ),
    },
    {
        "question": "What is supervised learning?",
        "reference": (
            "Supervised learning is learning a function that maps inputs "
            "to outputs from a set of labeled training examples."
        ),
    },
    {
        "question": "What is unsupervised learning?",
        "reference": (
            "Unsupervised learning is learning patterns or structure "
            "from data that has no labeled outputs."
        ),
    },
    {
        "question": "What is reinforcement learning?",
        "reference": (
            "Reinforcement learning is learning what actions to take, "
            "given a state, in order to maximize a numerical reward signal over time."
        ),
    },
    {
        "question": "What are the main applications of machine learning?",
        "reference": (
            "Machine learning is applied in areas such as data mining, "
            "speech and image recognition, fraud detection, autonomous "
            "vehicles, and information-filtering systems."
        ),
    },
]


In [40]:
eval_rows = []

for item in eval_set:
    answer, sources = rag(item["question"])

    eval_rows.append({
        "user_input": item["question"],
        "response": answer,
        "retrieved_contexts": [doc.page_content for doc in sources],
        "reference": item["reference"],
    })

print(f"Evaluation samples: {len(eval_rows)}")


Evaluation samples: 5


## 19. Connect DeepEval to local Qwen3 4B

DeepEval can use an Ollama-served model directly through `OllamaModel`.

This replaces the previous Ragas/OpenAI-compatible workaround.

The evaluator is configured entirely through the local Ollama endpoint.


In [41]:
from deepeval.models import OllamaModel

evaluator_model = OllamaModel(
    model=LOCAL_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

print("DeepEval evaluator:", LOCAL_MODEL)
print("Base URL:", OLLAMA_BASE_URL)


DeepEval evaluator: qwen3:4b-instruct
Base URL: http://localhost:11434


## 20. DeepEval RAG metrics

We use four native DeepEval RAG metrics:

- **Faithfulness:** checks whether the generated answer is supported by the retrieved context.
- **Answer Relevancy:** checks whether the answer addresses the question.
- **Contextual Precision:** checks whether relevant retrieved chunks are ranked higher.
- **Contextual Recall:** checks whether the retrieved context contains information needed for the reference answer.

Faithfulness and Answer Relevancy are referenceless. Contextual Precision and Contextual Recall use the expected/reference answer.


In [42]:
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

faithfulness_metric = FaithfulnessMetric(
    model=evaluator_model,
    threshold=0.5,
    include_reason=True,
)

answer_relevancy_metric = AnswerRelevancyMetric(
    model=evaluator_model,
    threshold=0.5,
    include_reason=True,
)

contextual_precision_metric = ContextualPrecisionMetric(
    model=evaluator_model,
    threshold=0.5,
    include_reason=True,
)

contextual_recall_metric = ContextualRecallMetric(
    model=evaluator_model,
    threshold=0.5,
    include_reason=True,
)

metrics = {
    "faithfulness": faithfulness_metric,
    "answer_relevancy": answer_relevancy_metric,
    "contextual_precision": contextual_precision_metric,
    "contextual_recall": contextual_recall_metric,
}

print("DeepEval metrics ready.")


DeepEval metrics ready.


## 21. Run one evaluation first

We evaluate a single row before running the complete set.

This makes it easier to catch a local-model or structured-output problem before spending time evaluating every sample.


In [43]:
from deepeval.test_case import LLMTestCase

row = eval_rows[0]

test_case = LLMTestCase(
    input=row["user_input"],
    actual_output=row["response"],
    expected_output=row["reference"],
    retrieval_context=row["retrieved_contexts"],
)

faithfulness_metric.measure(test_case)

print("Question:", row["user_input"])
print("Faithfulness score:", faithfulness_metric.score)
print("Reason:", faithfulness_metric.reason)


Output()

Question: What is machine learning?
Faithfulness score: 1.0
Reason: The faithfulness score is 1.00 because there are no contradictions between the actual output and the retrieval context.


## 22. Run the complete DeepEval evaluation

Each test case is evaluated with the four RAG metrics.

We run the metrics directly so the notebook keeps the evaluation flow explicit and easy to debug.


In [44]:
import pandas as pd
from deepeval.test_case import LLMTestCase

results = []

for row in eval_rows:
    print(f"Evaluating: {row['user_input']}")

    test_case = LLMTestCase(
        input=row["user_input"],
        actual_output=row["response"],
        expected_output=row["reference"],
        retrieval_context=row["retrieved_contexts"],
    )

    row_result = {
        "question": row["user_input"],
    }

    for name, metric in metrics.items():
        metric.measure(test_case)
        row_result[name] = float(metric.score)

    results.append(row_result)

results_df = pd.DataFrame(results)
results_df


Output()

Evaluating: What is machine learning?


Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,question,faithfulness,answer_relevancy,contextual_precision,contextual_recall
0,What is machine learning?,1.0,1.0,1.000000,0.500
1,What is supervised learning?,1.0,1.0,0.583333,0.250
2,What is unsupervised learning?,1.0,1.0,1.000000,1.000
3,What is reinforcement learning?,1.0,1.0,1.000000,0.375
4,What are the main applications of machine lear...,1.0,1.0,1.000000,1.000


## 23. Average DeepEval scores

Scores are between 0 and 1, where higher is generally better.

Because Qwen3 4B is also the evaluator, these scores should be treated as **model-based evaluation signals**, not absolute ground truth. They are most useful for comparing changes within this RAG project.


In [45]:
score_columns = [
    "faithfulness",
    "answer_relevancy",
    "contextual_precision",
    "contextual_recall",
]

results_df[score_columns].mean().sort_values(ascending=False)


faithfulness            1.000000
answer_relevancy        1.000000
contextual_precision    0.916667
contextual_recall       0.625000
dtype: float64

## 24. What each part of the project is doing

| Component | Purpose |
|---|---|
| BGE embeddings | Converts chunks and queries into vectors |
| ChromaDB | Stores vectors and performs dense retrieval |
| BM25 | Finds exact lexical matches |
| RRF | Combines dense + BM25 rankings |
| Cross-encoder | Reranks the strongest candidates |
| Qwen3 4B + Ollama | Generates the final answer locally |
| DeepEval + Qwen3 4B | Evaluates the generated answer and retrieved context locally |

### Important distinction

The **embedding model is not the generation model**.

BGE is used for retrieval.

Qwen3 4B is used for generation and as the DeepEval evaluator.

The notebook does not import or configure OpenAI.


## 25. Final pipeline

```text
                         ┌── BGE Embeddings ──→ ChromaDB ──→ Dense Search ──┐
PDF → Cleaning → Chunks ─┤                                                   ├→ RRF
                         └── BM25 ───────────────→ Keyword Search ───────────┘
                                                        ↓
                                                  Cross Encoder
                                                        ↓
                                               Top 4 Contexts
                                                        ↓
                                              Qwen3 4B / Ollama
                                                        ↓
                                                     Answer
                                                        ↓
                                              DeepEval / Qwen3 4B
                                                        ↓
                 Faithfulness / Answer Relevancy / Contextual Precision / Contextual Recall
```

### What changed from the previous notebook

- TinyLlama generation → **Qwen3 4B through Ollama**
- Ragas evaluation → **DeepEval native RAG metrics**
- Removed the previous cloud/API-based evaluation workaround
- Removed duplicated and obsolete evaluation cells
- Kept the existing PDF extraction, structure-aware chunking, ChromaDB, BM25, RRF and cross-encoder reranking pipeline


## 26. If DeepEval fails

Check these first:

1. Make sure Ollama is running.
2. Confirm the model exists with `ollama list`.
3. Confirm the model name is exactly `qwen3:4b`.
4. Test it with `ollama run qwen3:4b`.
5. Run the single Faithfulness sanity-check cell before running the full evaluation.

The generator and evaluator are both local, so an OpenAI API key is not required.
